# 03 — Customer Analysis
Repeat purchasing, customer value, and RFM segmentation using `customer_unique_id`.

In [ ]:
from pathlib import Path
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

ROOT = Path.cwd()
if ROOT.name == 'notebooks': ROOT = ROOT.parent
DASH = ROOT / 'data' / 'processed' / 'dashboard'
customers = pd.read_parquet(DASH / 'customer_features.parquet')
segments = pd.read_parquet(DASH / 'customer_segments.parquet')

In [ ]:
pd.Series({
    'unique_customers': customers['customer_unique_id'].nunique(),
    'repeat_customers': int(customers['repeat_customer'].sum()),
    'repeat_rate': customers['repeat_customer'].mean(),
    'average_historical_value': customers['total_spend'].mean(),
})

In [ ]:
rfm = customers.loc[customers['delivered_orders'].gt(0), ['recency', 'frequency', 'monetary']]
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
for column, axis in zip(rfm.columns, axes):
    sns.histplot(rfm[column].clip(upper=rfm[column].quantile(.99)), bins=40, ax=axis)
    axis.set_title(column.title())
plt.tight_layout();

In [ ]:
segments.groupby('segment').agg(
    customers=('customer_unique_id', 'nunique'),
    recency=('recency', 'mean'), frequency=('frequency', 'mean'), monetary=('monetary', 'mean')
).sort_values('monetary', ascending=False)